Evaluates models produced by trainer-v8, trained on versions of the Van Der Pol system

Made especially to test concentric ellipses of outside initial conditions

In [ ]:
# dependencies
from pathlib import Path
import re
import logging
import itertools
from IPython.display import display, Image 
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
import torch
import torch.nn as nn

# Set up standard logging instead of raw prints/warnings
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# global variables
WEIGHTS_FOLDER = 'trainer8-outsidedata-t4/' 
if WEIGHTS_FOLDER is not None:
    WEIGHTS_FILES = ['../model_weights/' + WEIGHTS_FOLDER + f.name for f in Path('../model_weights/' + WEIGHTS_FOLDER).iterdir() if f.is_file and 'epoch' in f.name]
else:
    WEIGHTS_FILES = ['../model_weights/trainer7-fouriersp-200ep-normalinit-0.0001std-t3/epoch200.pth']

SAVE = True
OVERWRITE = True
SIMPLY_LOAD = True
CALC_BLOWUP = False
EPOCHS_TO_SHOW = None #this gets set later on
MU = 10 # mu in vdp equation

# variable maps 
file_map = {
    'complete': {
        'inside': '../data/uhist-inside.hdf5',
        'outside': '../data/uhist-outside.hdf5',
        'all': '../data/uhist-all.hdf5'
    },
    'damped': {
        'inside': None,
        'outside': None,
        'all': '../data/uhist-damped-all.hdf5'
    }
}
series_length_map = {
    'complete': 400,
    'damped': 400
}
write_mode_map = {
    False: 'a',
    True: 'w'
}

In [3]:
# HELPER FUNCTIONS

def slideshow(epoch_to_image_path, epochs_to_show):
    """Makes and displays a slideshow of images"""
    # This function only loads the pre-saved image from disk, making it blazing fast
    def display_saved_plot(chosen_epoch):
        img_path = epoch_to_image_path[chosen_epoch]
        display(Image(filename=img_path))

    # --- UI Layout Elements ---
    epoch_slider = widgets.SelectionSlider(
        options=epochs_to_show,        
        value=epochs_to_show[0],       
        description='Epoch:',
        continuous_update=True # Can be True now because loading images is instantaneous!
    )

    def on_prev_clicked(b):
        current_index = epochs_to_show.index(epoch_slider.value)
        if current_index > 0:
            epoch_slider.value = epochs_to_show[current_index - 1]

    def on_next_clicked(b):
        current_index = epochs_to_show.index(epoch_slider.value)
        if current_index < len(epochs_to_show) - 1:
            epoch_slider.value = epochs_to_show[current_index + 1]

    prev_button = widgets.Button(description='', icon='arrow-left', layout=widgets.Layout(width='50px'))
    next_button = widgets.Button(description='', icon='arrow-right', layout=widgets.Layout(width='50px'))

    prev_button.on_click(on_prev_clicked)
    next_button.on_click(on_next_clicked)

    # Link the widget to our lightweight image loader function
    plot_output = widgets.interactive_output(display_saved_plot, {'chosen_epoch': epoch_slider})

    # Assemble the UI
    ui = widgets.VBox([
        widgets.HBox([prev_button, epoch_slider, next_button]), 
        plot_output
    ])

    display(ui)

def compute_paired_dtw_distances(traj_group_a, traj_group_b):
    """
    Computes multivariate DTW distance for each corresponding pair.
    
    traj_group_a: shape (N_pairs, T_a, 2)
    traj_group_b: shape (N_pairs, T_b, 2)
    Returns: array of distances of shape (N_pairs,)
    """
    n_pairs = len(traj_group_a)
    distances = np.zeros(n_pairs)
    
    for i in range(n_pairs):
        # fastdtw accepts (T, D) arrays where D=2 for 2D trajectories
        dist, _ = fastdtw(traj_group_a[i], traj_group_b[i], dist=euclidean)
        distances[i] = dist
        
    return distances


def paired_dtw_permutation_test(traj_group_a, traj_group_b, n_permutations=2000, seed=42):
    """
    Performs a paired permutation test on 2D trajectories using multivariate DTW.
    """
    np.random.seed(seed)
    n_pairs = len(traj_group_a)
    
    # 1. Compute trajectory error vector for each of the 100 pairs
    # (e.g., mean Euclidean error across time steps for trajectory i)
    # Shape: (100,)
    diffs = np.mean(np.linalg.norm(traj_group_a - traj_group_b, axis=2), axis=1)
    
    obs_mean = np.mean(diffs)
    
    # 2. Correct Permutation: Randomize the SIGNS (+1 or -1) 
    # Null Hypothesis: True difference is centered at 0
    # Shape: (n_perms, 100)
    random_signs = np.random.choice([-1, 1], size=(n_permutations, n_pairs))
    
    # Compute permuted means under H0
    perm_means = np.mean(diffs * random_signs, axis=1)
    
    # 3. Two-tailed p-value (or upper-tailed)
    p_value = np.mean(np.abs(perm_means) >= np.abs(obs_mean))
    
    return obs_mean, p_value


In [4]:
# DEFINE MODEL ARCHITECTURES

class neuralODE8(nn.Module):
    def __init__(self, act_layer=nn.Sigmoid, dim_inout=2, init_std=0.01):
        super(neuralODE8, self).__init__()

        # save the initial standard deviation to the reigster buffer
        self.register_buffer('init_std', torch.tensor([init_std]))

        #save space in the register buffer for data means and standard deviations (1 per column)
        self.register_buffer('data_mean', torch.zeros(dim_inout)) # placeholder mean
        self.register_buffer('data_std', torch.ones(dim_inout)) # placeholder stdev
        #self.register_buffer('layer_init_record', torch.ones(3))

        # use a linear & sigmoid fully connected nn
        self.net = nn.Sequential(
            nn.Linear(dim_inout, 200),
            act_layer(),
            nn.Linear(200, 200),
            act_layer(),
            nn.Linear(200, 200),
            act_layer(),
            nn.Linear(200, dim_inout)
        )
        self.init_weights()

    # initialize model weights & biases
    def init_weights(self):
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                # initialize based on inputted stdev
                nn.init.normal_(m.weight, mean=0, std=0)#self.init_std.item())
                nn.init.constant_(m.bias, 0.0)

    # forward function adapted for scipy integrate instead of odeint
    def forward(self, t, x):
        x = torch.tensor(x)
        x = x.type(torch.FloatTensor)
        return self.net(x).detach().numpy()

    # sets the data_mean and data_std in the register buffer
    def set_normalization_stats(self, mean, std):
        self.data_mean = mean
        self.data_std = std

In [5]:
# DEFINE MODEL WRAPPER OBJECT FOR OOP-based access of model & metadata later

class ModelWrapper:
    def __init__(self, weights_file: str, simply_load: bool, overwrite: bool):
        self.weights_file = Path(weights_file)
        self.clean_weights_file = Path(Path(weights_file).parent.name) / Path(weights_file).stem
        self.simply_load = simply_load
        self.overwrite = overwrite
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        metadata = self._parse_metadata()
        self.trainer_version = metadata.get('trainer_version', None)
        self.equation = metadata.get('equation', 'complete')
        self.observable_data = metadata.get('observable_data', None)
        self.init_std = metadata.get('init_std', 0.01)
        self.normalized = metadata.get('normalized', False)
        self.trial = metadata.get('trial', 1)
        self.epoch = metadata.get('epoch', 0)
        
        self.model = self._initialize_and_load_model()

        self.save_dirs = self._make_save_dirs()

        self._pred = None
        self._pred1 = None
        self._pred2 = None
        self._pred3 = None

    def _parse_metadata(self):
        """Helper method to parse the metadata from the file name."""
        file_path = self.weights_file.as_posix()
        meta = {}
        
        # parse trainer version
        if 'trainer8' in file_path: meta['trainer_version'] = 8
        else:
            raise ValueError(f"Couldn't parse trainer version from path: {file_path}")

        # parse equation
        if 'dampedeq' in file_path: meta['equation'] = 'damped'

        # parse observable data
        if 'insidedata' in file_path: meta['observable_data'] = 'inside'
        elif 'outsidedata' in file_path: meta['observable_data'] = 'outside'
        elif 'alldata' in file_path: meta['observable_data'] = 'all'
        else:
            raise ValueError(f"Couldn't parse observable data from path: {file_path}")

        # parse initial standard deviation
        stds = ['0.0001', '0.001', '0.1', '1.0']
        for std in stds:
            if std + 'std' in file_path: 
                meta['init_std'] = float(std)
                break

        # parse normalization
        if 'normalized' in file_path: meta['normalized'] = True

        # parse trial number
        for trial in range(1, 10):
            trial_str = 't' + str(trial)
            if trial_str in file_path:
                meta['trial'] = trial

        # parse epoch number
        if 'epoch' in file_path:
            split_file_path = file_path.split('/')
            meta['epoch'] = int(re.findall(r'\d+', split_file_path[-1])[0])
        else:
            logger.warning(f"Couldn't parse epoch number from file path: {file_path}")

        return meta

    def _initialize_and_load_model(self):
        """Instantiates the ODEModel and loads the weights into it."""
        # make sure the file actually exists
        if not self.weights_file.exists():
            raise FileNotFoundError(f"No weights file found at {self.weights_file}")
        
        # create the appropriate model based on the various architectures
        if self.trainer_version == 8:
            model = neuralODE8(init_std=self.init_std) 
        
        # load the trained model weights into the model
        try:
            state_dict = torch.load(self.weights_file, map_location=self.device)
            model.load_state_dict(state_dict)
        except Exception as e:
            logger.error(f"Failed to load state dict: for {self.weights_file}")
            raise e

        # put model on desired device and set to evaluation mode
        model.to(self.device)
        model.eval()
        
        return model

    @staticmethod
    def iger(a, b, num):
        """Generates evenly space initial conditions along three concentric ellispses, and rejects if they're inside the curve:
        $6.5\\left(x+2\\right)\\sin\\left(0.7\\left(x+2\\right)\\right)-x-2$ or its 180 degree rotated counterpart"""

        f = lambda q: 6.5 * (q+2) * np.sin(0.7 * (q+2)) - q - 2

        initial_conditions = []
        thetas = np.linspace(0, 2*np.pi, num, endpoint=False)
        for theta in thetas:
            x = a*np.cos(theta)
            y = b*np.sin(theta)

            if not (y > -f(-x) and y < f(x)):
                initial_conditions.append([x, y])

        initial_conditions = np.array(initial_conditions)

        return initial_conditions

    @property
    def pred(self):
        """Lazy loading prediction"""
        if self._pred is None:
            if self.simply_load:
                self._pred = pd.read_hdf(f'../predictions/{self.clean_weights_file}.hdf5', key='df').to_numpy()
            else:
                
                self._pred = self._make_predictions()
        return self._pred

    @property
    def pred1(self):
        """Lazy loading prediction"""
        if self._pred1 is None:
            if self.simply_load:
                self._pred = pd.read_hdf(f'../predictions_extra/one/{self.clean_weights_file}.hdf5', key='df').to_numpy()
            else:
                self.A, self.B = 2, 15
                aa = self.A*np.array([0.75, 1.2, 1.6, 2.0])
                bb = self.B*np.array([0.75, 1.2, 1.6, 2.0])
                initial_conditions = np.concatenate((self.iger(aa[0], bb[0], 10), self.iger(aa[1], bb[1], 10), self.iger(aa[2], bb[2], 12), self.iger(aa[3], bb[3], 14)))
                self._pred1 = self._make_predictions(initial_coniditions=initial_conditions)
        return self._pred1

    @property
    def pred2(self):
        """Lazy loading prediction"""
        if self._pred2 is None:
            if self.simply_load:
                self._pred2 = pd.read_hdf(f'../predictions_extra/two/{self.clean_weights_file}.hdf5', key='df').to_numpy()
            else: 
                self._pred2 = self._make_predictions()
        return self._pred2

    @property
    def pred3(self):
        """Lazy loading prediction"""
        if self._pred3 is None:
            if self.simply_load:
                self._pred3 = pd.read_hdf(f'../predictions_extra/three/{self.clean_weights_file}.hdf5', key='df').to_numpy()
            else: 
                self._pred3 = self._make_predictions()
        return self._pred3

    def _make_predictions(self, initial_coniditions=None):
        """Makes predictions about the K-S system based on the model"""

        if initial_coniditions is None:
            initial_coniditions = []
            row_num = 0
            while row_num < 40000:
                initial_coniditions.append(pd.read_hdf(file_map[self.equation][self.observable_data], key='df').iloc[row_num].to_numpy())
                row_num += series_length_map[self.equation]

        pred = np.zeros((1,2))

        for initial_condition in initial_coniditions:
            # set evaluation times for solve_ivp
            t_f = series_length_map[self.equation]
            dt = 0.05
            ts_eval = np.arange(0.0, t_f*dt, dt)

            if self.normalized:
                # get the mean and the standard deviation from the model's register buffer
                mean = (self.model.data_mean).numpy()
                std = self.model.data_std.numpy()
                # normalize the initial condition
                initial_condition = (initial_condition - mean) / std

            this_pred = solve_ivp(self.model.forward, [0,ts_eval[-1]], initial_condition, t_eval=ts_eval).y.T
            if self.normalized: 
                pred = (pred * std) + mean
            pred = np.concatenate((pred, this_pred))

        pred = np.delete(pred, (0), axis=0)

        return pred
    
    def save_predictions(self, dir_key='predictions'):
        """Saves calculated information about Fourier mode amplitudes"""
        write_mode = write_mode_map[self.overwrite]

        if self.pred is not None:
            df = pd.DataFrame(self.pred)
            save_name = self.save_dirs[dir_key]
            df.to_hdf(save_name, key='df', mode=write_mode, complib='blosc', complevel=9)
        else: 
            logger.warning(f"There is nothing to save at {save_name}")
    
    def _make_save_dirs(self):
        dirs = {
            'phase_portriats': Path('../' + 'phase_portraits' + '/' + self.clean_weights_file.as_posix() + '.png'),
            'slope_fields_raw': Path('../' + 'slope_fields_raw' + '/' + self.clean_weights_file.as_posix() + '.png'),
            'slope_fields_scaled': Path('../' + 'slope_fields_scaled' + '/' + self.clean_weights_file.as_posix() + '.png'), 
            'predictions': Path('../' + 'predictions' + '/' + self.clean_weights_file.as_posix() + '.hdf5'),
            'predictions_extra_one': Path('../' + 'predictions_extra/one/' + self.clean_weights_file.as_posix() + '.hdf5'),
            'predictions_extra_two': Path('../' + 'predictions_extra/two/' + self.clean_weights_file.as_posix() + '.hdf5'),
            'predictions_extra_three': Path('../' + 'predictions_extra/three/' + self.clean_weights_file.as_posix() + '.hdf5'),
            'phase_portraits_extra_one': Path('../' + 'phase_portraits_extra/one/' + self.clean_weights_file.as_posix() + '.png'),
            'phase_portraits_extra_two': Path('../' + 'phase_portraits_extra/two/' + self.clean_weights_file.as_posix() + '.png'),
            'phase_portraits_extra_three': Path('../' + 'phase_portraits_extra/three/' + self.clean_weights_file.as_posix() + '.png')
            }

        # check to make sure the save directories don't exist before making them
        for v in dirs.values():
            if not self.overwrite and v.exists():
                raise FileExistsError(f"{v} already exists. Change OVERWRITE to True if you want to overwrite the existing version.")
            v.parent.mkdir(parents=True, exist_ok=True)
        return dirs

wrapped_models = []
for file in WEIGHTS_FILES:
    wrapped_models.append(ModelWrapper(file, SIMPLY_LOAD, not SAVE^OVERWRITE))

# create/ assign variables to be used for plotting
epoch_to_model_map = {wm.epoch: (idx, wm) for idx, wm in enumerate(wrapped_models)}
EPOCHS_TO_SHOW = [wrapped_model.epoch for wrapped_model in wrapped_models]

In [6]:
# USE MODEL TO MAKE PREDICTIONS

for wrapped_model in wrapped_models:
    try:
        #wrapped_model.pred
        wrapped_model.pred1
    except Exception as e:
        logger.exception(f"Failed to make predictions for model at {wrapped_model.weights_file.as_posix()}")
        raise

    if not wrapped_model.simply_load: wrapped_model.save_predictions(dir_key='predictions_extra_one')

KeyboardInterrupt: 

In [ ]:
# SAVING AND PLOTTING FIGURES

import matplotlib.patches as mpatches

# Dictionary to store the absolute file path for each epoch
if SAVE: epoch_to_image_path = {} 

for idx, wrapped_model in enumerate(wrapped_models):
    if SAVE:
        file_path = wrapped_model.save_dirs['phase_portraits_extra_one']
        epoch_to_image_path[wrapped_model.epoch] = str(file_path)

    # plot setup
    plt.figure(figsize=(12, 6))

    # this stuff is only needed if you're doing the concentric ellipses
    aa = 2*np.array([0.75, 1.2, 1.6, 2.0])
    bb = 15*np.array([0.75, 1.2, 1.6, 2.0])
    initial_conditions = np.concatenate((wrapped_model.iger(aa[0], bb[0], 10), wrapped_model.iger(aa[1], bb[1], 10), wrapped_model.iger(aa[2], bb[2], 12), wrapped_model.iger(aa[3], bb[3], 14)))
    # plot ellipses of initial conditions (if you're doing that)
    for a, b in zip(aa, bb):
        ellipse = mpatches.Ellipse((0,0), 2*a, 2*b, fill=False, edgecolor='lightgray', linewidth=2, linestyle=':')
        plt.gca().add_patch(ellipse)

    row_num = 0    
    
    while row_num < 40000:
        u = wrapped_model.pred1[row_num:(row_num + series_length_map[wrapped_model.equation]), :]
        plt.plot(u[:, 0], u[:, 1], alpha=0.6)
        row_num += series_length_map[wrapped_model.equation]

    # plot labels
    plt.suptitle(
        f"Model {idx+1}: {wrapped_model.equation} eqn | {wrapped_model.observable_data} I.C.s | trial {wrapped_model.trial} | {wrapped_model.epoch} epochs", 
        fontsize=14, fontweight='bold'
        )
    plt.title('Phase portraits for concentric initial conditions', fontsize=10)
    plt.xlabel('$u_0$', fontsize=12)
    plt.ylabel('$u_1$', fontsize=12)
    plt.tight_layout()

    if SAVE:
        plt.savefig(file_path, dpi=150)
        plt.close()
    else:
        plt.show()

    print(wrapped_model.epoch)

if SAVE:
    print("All plots saved successfully!")
    slideshow(epoch_to_image_path, EPOCHS_TO_SHOW)